# run_e30
Phase 2 Stage 3 E30 CPU worker: head x pooling swap at fixed midpoint `image_caption` (gated on `phase2_e29_gate` `decisions['E30']['decision'] == 'OPEN'`).

- Configure only cell 3 (`MODEL = 'qwen'` or `'gemma'`, `PLAN_REF`, `E29_GATE_REF`, optional resume refs).
- Uses the pinned Python 3.11/3.12 CPU runtime (`requirements-stage2-cpu.txt`) with single-threaded BLAS/OpenMP.
- Verifies identical depth, hook, dimension, `model_plan`, and 360-image SCREEN row keys across `last_caption` and `caption_mean`, asserts `en->en` matched-arm reproduction (`atol=0, rtol=0`), scores ONLY the 2 crossed arms (8 blocks / 32 routes), and evaluates all 4 arms on the frozen `(2000, 360)` bootstrap. Phase 1 stays strictly read-only and `CONFIRM` stays closed.


In [ ]:
import os, sys, subprocess
from pathlib import Path

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/Research-phase2-e30')
CODE_REF = 'phase2'  # Set to the approved Phase 2 branch or commit SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage 3 E30 settings (edit only this cell).
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
MODEL = 'qwen'              # 'qwen' or 'gemma' (run one runtime per model).
PLAN_REF = None             # Required: {'path': 'phase2/plans/.../_COMPLETE.json', 'sha256': '...'} or Path.
E29_GATE_REF = None         # Required: model's phase2_e29_gate {'path': 'phase2/results/e29_gates/.../_COMPLETE.json', 'sha256': '...'} or Path.
RESUME_CROSSED_REFS = None  # Optional dict {'head_last_on_mean:en': {'path': ..., 'sha256': ...}, ...} or JSON Path.
RESUME_ANALYSIS_REF = None  # Optional explicit {'path': ..., 'sha256': ...} or Path to phase2_analysis_block _COMPLETE.json.
OUTPUT_REF = None           # Optional mounted JSON path (e.g. PHASE1_ROOT / f'phase2/results/e30_{MODEL}_ref.json').


In [ ]:
import json, signal

if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 2 checkout.')
if MODEL not in ('qwen', 'gemma'):
    raise RuntimeError("Set MODEL to 'qwen' or 'gemma'.")
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Missing mounted PHASE1_ROOT directory: {PHASE1_ROOT}')
if PLAN_REF is None:
    raise RuntimeError('Set PLAN_REF to the verified Stage 1 phase2_plan {path, sha256} or Path.')
if E29_GATE_REF is None:
    raise RuntimeError('Set E29_GATE_REF to the verified Stage 2 phase2_e29_gate {path, sha256} or Path for MODEL.')

def _format_ref(ref, label):
    if isinstance(ref, dict):
        target = PHASE1_ROOT / ref['path']
        if not target.is_file():
            raise RuntimeError(f'Missing {label} manifest: {target}')
        return json.dumps(ref)
    target = Path(ref)
    if not target.is_file():
        raise RuntimeError(f'Missing {label} manifest: {target}')
    return str(target)

def _format_mapping(mapping, label):
    if isinstance(mapping, dict):
        for key, ref in mapping.items():
            if ref is not None:
                _format_ref(ref, f'{label}[{key}]')
        return json.dumps(mapping)
    target = Path(mapping)
    if not target.is_file():
        raise RuntimeError(f'Missing {label} JSON file: {target}')
    return str(target)

command = [
    sys.executable, '-u', '-m', 'src.phase2.run_e30',
    '--phase1-root', str(PHASE1_ROOT),
    '--plan', _format_ref(PLAN_REF, 'phase2_plan'),
    '--e29-gate', _format_ref(E29_GATE_REF, 'phase2_e29_gate'),
    '--model', MODEL,
    '--storage-scope', 'persistent-drive',
]
if RESUME_CROSSED_REFS is not None:
    command += ['--resume-crossed-refs', _format_mapping(RESUME_CROSSED_REFS, 'resume_crossed_refs')]
if RESUME_ANALYSIS_REF is not None:
    command += ['--resume-analysis-ref', _format_ref(RESUME_ANALYSIS_REF, 'resume_analysis')]
if OUTPUT_REF is not None:
    command += ['--output-ref', str(OUTPUT_REF)]

process = subprocess.Popen(
    command, cwd=REPO, env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
blockers = []
events = []
try:
    for line in process.stdout:
        print(line, end='', flush=True)
        try:
            payload = json.loads(line)
            if isinstance(payload, dict):
                events.append(payload)
                if payload.get('status') == 'blocked':
                    blockers.extend(payload.get('blockers', []))
        except json.JSONDecodeError:
            pass
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    raise
code = process.wait()
if code != 0:
    detail = '; '.join(blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'E30 worker blocked/failed for {MODEL}: {detail}')

completed = [e for e in events if e.get('event') == 'e30_complete']
if completed:
    print('\n=== E30 CROSSED & ANALYSIS REFERENCES ===')
    print(json.dumps(completed[-1], indent=2))
